# Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


# Load Dataset

In [2]:
file_path = "D:\Samina Tata Project/Coding/Dataset/Crimes_-_One_year_prior_to_present_20260831.csv"

df = pd.read_csv(file_path)

print("Raw dataset loaded.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\shrijal\AppData\Local\Temp\ipykernel_15164\3036300694.py:1: SyntaxWarning: invalid escape sequence '\S'
  file_path = "D:\Samina Tata Project/Coding/Dataset/Crimes_-_One_year_prior_to_present_20260831.csv"


Raw dataset loaded.
Rows: 229275
Columns: 17


# Make column names easier to work with

In [3]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_")

print(df.columns.tolist())

['CASE#', 'DATE_OF_OCCURRENCE', 'BLOCK', 'IUCR', 'PRIMARY_DESCRIPTION', 'SECONDARY_DESCRIPTION', 'LOCATION_DESCRIPTION', 'ARREST', 'DOMESTIC', 'BEAT', 'WARD', 'FBI_CD', 'X_COORDINATE', 'Y_COORDINATE', 'LATITUDE', 'LONGITUDE', 'LOCATION']


# Convert The date

In [4]:
df["DATE_OF_OCCURRENCE"] = pd.to_datetime(
    df["DATE_OF_OCCURRENCE"],
    errors="coerce"
)

print(df["DATE_OF_OCCURRENCE"].dtype)
print(df["DATE_OF_OCCURRENCE"].head())

datetime64[ns]
0   2025-08-24 05:03:00
1   2025-08-24 05:25:00
2   2025-08-24 05:30:00
3   2025-08-24 05:30:00
4   2025-08-24 05:32:00
Name: DATE_OF_OCCURRENCE, dtype: datetime64[ns]


# Check whether date conversion created missing values

In [5]:
missing_dates = df["DATE_OF_OCCURRENCE"].isna().sum()

print("Invalid/missing dates:", missing_dates)

Invalid/missing dates: 0


# Remove Duplicate Records

In [6]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)

Rows before removing duplicates: 229275
Rows after removing duplicates: 229273
Duplicates removed: 2


# Remove records without coordinates

In [7]:
before = len(df)

df = df.dropna(
    subset=["LATITUDE", "LONGITUDE"]
)

after = len(df)

print("Rows before removing missing coordinates:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before removing missing coordinates: 229273
Rows after: 229233
Rows removed: 40


# Check geographic boundaries

#### Inspect how many of our records fall outside Chicago boundary in the dataset

In [8]:
outside_boundary = (
    (df["LATITUDE"] < 41.6) |
    (df["LATITUDE"] > 42.1) |
    (df["LONGITUDE"] < -87.9) |
    (df["LONGITUDE"] > -87.5)
)

print("Records outside research-paper boundary:",
      outside_boundary.sum())

print("Percentage:",
      round(outside_boundary.mean() * 100, 4), "%")

Records outside research-paper boundary: 638
Percentage: 0.2783 %


# Remove data outside the boundry

In [ ]:
lat_min = 41.6
lat_max = 42.1
lon_min = -87.9
lon_max = -87.5

before = len(df)

df = df[
    (df["LATITUDE"] >= lat_min) &
    (df["LATITUDE"] <= lat_max) &
    (df["LONGITUDE"] >= lon_min) &
    (df["LONGITUDE"] <= lon_max)
].copy()

after = len(df)

print("Rows before boundary filtering:", before)
print("Rows after boundary filtering:", after)
print("Rows removed:", before - after)
print("Percentage removed:", round((before - after) / before * 100, 4), "%")

Rows before boundary filtering: 229233
Rows after boundary filtering: 228595
Rows removed: 638
Percentage removed: 0.2783 %


# Validation after data removed

In [10]:
print("Latitude range after filtering:")
print(df["LATITUDE"].min(), "to", df["LATITUDE"].max())

print("\nLongitude range after filtering:")
print(df["LONGITUDE"].min(), "to", df["LONGITUDE"].max())

Latitude range after filtering:
41.644589713 to 42.022547568

Longitude range after filtering:
-87.899493147 to -87.524529378


# Mark missing location description as unknown

In [11]:
# Handle missing location descriptions

missing_location_before = df["LOCATION_DESCRIPTION"].isna().sum()

df["LOCATION_DESCRIPTION"] = (
    df["LOCATION_DESCRIPTION"]
    .fillna("UNKNOWN")
)

missing_location_after = df["LOCATION_DESCRIPTION"].isna().sum()

print("Missing LOCATION_DESCRIPTION before:", missing_location_before)
print("Missing LOCATION_DESCRIPTION after:", missing_location_after)

Missing LOCATION_DESCRIPTION before: 1037
Missing LOCATION_DESCRIPTION after: 0


# Check ward

In [12]:
print("Missing WARD values:", df["WARD"].isna().sum())

print("\nRecord with missing WARD:")
display(
    df[df["WARD"].isna()]
)

Missing WARD values: 1

Record with missing WARD:


,CASE#,DATE_OF_OCCURRENCE,BLOCK,IUCR,PRIMARY_DESCRIPTION,SECONDARY_DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,WARD,FBI_CD,X_COORDINATE,Y_COORDINATE,LATITUDE,LONGITUDE,LOCATION
4642,JJ407553,2025-08-31 12:00:00,026XX W 115TH ST,1187,DECEPTIVE PRACTICE,STATE BENEFITS FRAUD,RESIDENCE,N,N,2212,NaN,11,1160919.0,1828275.0,41.684467,-87.686554,"(41.684467361, -87.686554462)"


In [13]:
print("Missing WARD values:", df["WARD"].isna().sum())
print("Total records:", len(df))

print("\nMissing WARD record:")
display(df[df["WARD"].isna()])

Missing WARD values: 1
Total records: 228595

Missing WARD record:


,CASE#,DATE_OF_OCCURRENCE,BLOCK,IUCR,PRIMARY_DESCRIPTION,SECONDARY_DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,WARD,FBI_CD,X_COORDINATE,Y_COORDINATE,LATITUDE,LONGITUDE,LOCATION
4642,JJ407553,2025-08-31 12:00:00,026XX W 115TH ST,1187,DECEPTIVE PRACTICE,STATE BENEFITS FRAUD,RESIDENCE,N,N,2212,NaN,11,1160919.0,1828275.0,41.684467,-87.686554,"(41.684467361, -87.686554462)"


# Check current data types

In [14]:
df.dtypes

CASE#                            object
DATE_OF_OCCURRENCE       datetime64[ns]
BLOCK                            object
IUCR                             object
PRIMARY_DESCRIPTION              object
SECONDARY_DESCRIPTION            object
LOCATION_DESCRIPTION             object
ARREST                           object
DOMESTIC                         object
BEAT                              int64
WARD                            float64
FBI_CD                           object
X_COORDINATE                    float64
Y_COORDINATE                    float64
LATITUDE                        float64
LONGITUDE                       float64
LOCATION                         object
dtype: object

# Check date range again

In [15]:
print("Minimum date:", df["DATE_OF_OCCURRENCE"].min())
print("Maximum date:", df["DATE_OF_OCCURRENCE"].max())

Minimum date: 2025-08-24 05:03:00
Maximum date: 2026-08-23 00:00:00


# Check for invalid Coordinate

In [16]:
print("Invalid latitude values:",
      ((df["LATITUDE"] < 41.6) | (df["LATITUDE"] > 42.1)).sum())

print("Invalid longitude values:",
      ((df["LONGITUDE"] < -87.9) | (df["LONGITUDE"] > -87.5)).sum())

Invalid latitude values: 0
Invalid longitude values: 0


# Check for zero coordinate

In [17]:
print("Latitude = 0:",
      (df["LATITUDE"] == 0).sum())

print("Longitude = 0:",
      (df["LONGITUDE"] == 0).sum())

Latitude = 0: 0
Longitude = 0: 0


# Check categorical Column

In [18]:
categorical_columns = [
    "PRIMARY_DESCRIPTION",
    "SECONDARY_DESCRIPTION",
    "LOCATION_DESCRIPTION",
    "ARREST",
    "DOMESTIC",
    "FBI_CD"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print("Missing values:", df[column].isna().sum())
    print(df[column].value_counts().head(10))


--- PRIMARY_DESCRIPTION ---
Unique values: 31
Missing values: 0
PRIMARY_DESCRIPTION
THEFT                  49763
BATTERY                42093
CRIMINAL DAMAGE        25501
ASSAULT                20706
MOTOR VEHICLE THEFT    18114
OTHER OFFENSE          15632
DECEPTIVE PRACTICE     13232
BURGLARY               12917
NARCOTICS               6473
CRIMINAL TRESPASS       5477
Name: count, dtype: int64

--- SECONDARY_DESCRIPTION ---
Unique values: 309
Missing values: 0
SECONDARY_DESCRIPTION
SIMPLE                         29254
DOMESTIC BATTERY SIMPLE        18556
TO VEHICLE                     15080
$500 AND UNDER                 14133
AUTOMOBILE                     14128
OVER $500                      12433
RETAIL THEFT                   12090
TO PROPERTY                    10055
BURGLARY FROM MOTOR VEHICLE     7157
THEFT FROM MOTOR VEHICLE        5855
Name: count, dtype: int64

--- LOCATION_DESCRIPTION ---
Unique values: 126
Missing values: 0
LOCATION_DESCRIPTION
STREET                   

# Check Arrest

In [19]:
print(df["ARREST"].value_counts())

ARREST
N    193248
Y     35347
Name: count, dtype: int64


In [20]:
print("ARREST values:")
print(df["ARREST"].unique())

ARREST values:
['Y' 'N']


# Check domestic

In [21]:
print(df["DOMESTIC"].value_counts())

print("\nDOMESTIC unique values:")
print(df["DOMESTIC"].unique())

DOMESTIC
N    184611
Y     43984
Name: count, dtype: int64

DOMESTIC unique values:
['Y' 'N']


# check BEAT

In [22]:
print("Number of unique beats:", df["BEAT"].nunique())

print("\nBEAT range:")
print("Minimum:", df["BEAT"].min())
print("Maximum:", df["BEAT"].max())

print("\nMost common beats:")
print(df["BEAT"].value_counts().head(15))

Number of unique beats: 272

BEAT range:
Minimum: 111
Maximum: 2535

Most common beats:
BEAT
1834    2761
123     1908
1831    1730
421     1699
112     1613
1214    1587
631     1547
331     1545
222     1540
1832    1504
1232    1501
823     1496
624     1486
1224    1483
511     1443
Name: count, dtype: int64


# check Ward

In [23]:
print("Number of unique wards:", df["WARD"].nunique())

print("Minimum ward:", df["WARD"].min())
print("Maximum ward:", df["WARD"].max())

print("\nMissing wards:", df["WARD"].isna().sum())

Number of unique wards: 50
Minimum ward: 1.0
Maximum ward: 50.0

Missing wards: 1


# Check crime names for whitespace problems

In [24]:
text_columns = [
    "PRIMARY_DESCRIPTION",
    "SECONDARY_DESCRIPTION",
    "LOCATION_DESCRIPTION",
    "ARREST",
    "DOMESTIC",
    "FBI_CD"
]

for column in text_columns:
    leading_spaces = df[column].astype(str).str.startswith(" ").sum()
    trailing_spaces = df[column].astype(str).str.endswith(" ").sum()
    
    print(f"{column}:")
    print("  Leading spaces:", leading_spaces)
    print("  Trailing spaces:", trailing_spaces)

PRIMARY_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
SECONDARY_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
LOCATION_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
ARREST:
  Leading spaces: 0
  Trailing spaces: 0
DOMESTIC:
  Leading spaces: 0
  Trailing spaces: 0
FBI_CD:
  Leading spaces: 0
  Trailing spaces: 0


# Check empty strings

In [25]:
for column in text_columns:
    empty_count = (
        df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    print(f"{column}: {empty_count} empty strings")

PRIMARY_DESCRIPTION: 0 empty strings
SECONDARY_DESCRIPTION: 0 empty strings
LOCATION_DESCRIPTION: 0 empty strings
ARREST: 0 empty strings
DOMESTIC: 0 empty strings
FBI_CD: 0 empty strings


# Check duplicate CASE numbers

In [26]:
duplicate_cases = df["CASE#"].duplicated().sum()

print("Duplicate CASE# values:", duplicate_cases)

Duplicate CASE# values: 22


# Final duplicate check

In [27]:
print("Exact duplicate rows remaining:", df.duplicated().sum())

Exact duplicate rows remaining: 0


In [29]:
text_columns = [
    "PRIMARY_DESCRIPTION",
    "SECONDARY_DESCRIPTION",
    "LOCATION_DESCRIPTION",
    "ARREST",
    "DOMESTIC",
    "FBI_CD"
]

for column in text_columns:
    leading_spaces = df[column].astype(str).str.startswith(" ").sum()
    trailing_spaces = df[column].astype(str).str.endswith(" ").sum()
    
    print(f"{column}:")
    print("  Leading spaces:", leading_spaces)
    print("  Trailing spaces:", trailing_spaces)

PRIMARY_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
SECONDARY_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
LOCATION_DESCRIPTION:
  Leading spaces: 0
  Trailing spaces: 0
ARREST:
  Leading spaces: 0
  Trailing spaces: 0
DOMESTIC:
  Leading spaces: 0
  Trailing spaces: 0
FBI_CD:
  Leading spaces: 0
  Trailing spaces: 0


In [30]:
for column in text_columns:
    empty_count = (
        df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    print(f"{column}: {empty_count} empty strings")

PRIMARY_DESCRIPTION: 0 empty strings
SECONDARY_DESCRIPTION: 0 empty strings
LOCATION_DESCRIPTION: 0 empty strings
ARREST: 0 empty strings
DOMESTIC: 0 empty strings
FBI_CD: 0 empty strings


# Final Missing value check

In [31]:
# Final missing-value check

missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().sum() / len(df) * 100).round(4)
})

missing_summary = missing_summary[missing_summary["Missing Count"] > 0]

missing_summary

,Missing Count,Missing Percentage
WARD,1,0.0004


# Final Duplicate Check

In [32]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate CASE# values:", df["CASE#"].duplicated().sum())

Exact duplicate rows: 0
Duplicate CASE# values: 22


# Final Dataset Information

In [33]:
print("Final dataset shape:", df.shape)

print("\nDate range:")
print(df["DATE_OF_OCCURRENCE"].min())
print(df["DATE_OF_OCCURRENCE"].max())

print("\nGeographical range:")

print("Latitude:",
      df["LATITUDE"].min(),
      "to",
      df["LATITUDE"].max())

print("Longitude:",
      df["LONGITUDE"].min(),
      "to",
      df["LONGITUDE"].max())

Final dataset shape: (228595, 17)

Date range:
2025-08-24 05:03:00
2026-08-23 00:00:00

Geographical range:
Latitude: 41.644589713 to 42.022547568
Longitude: -87.899493147 to -87.524529378


# Save the cleaned data

In [35]:
import os

os.makedirs("../data", exist_ok=True)
os.makedirs("data", exist_ok=True)

cleaned_file = "data/chicago_crime_cleaned.csv"

df.to_csv(
    cleaned_file,
    index=False
)

print("Cleaned dataset saved to:")
print(cleaned_file)

Cleaned dataset saved to:
data/chicago_crime_cleaned.csv


In [36]:
test_df = pd.read_csv(cleaned_file)

print("Saved file loaded successfully.")
print("Rows:", test_df.shape[0])
print("Columns:", test_df.shape[1])

Saved file loaded successfully.
Rows: 228595
Columns: 17
